# 07 — Baselines: SVM + Random Forest with LOSO-CV

Handcrafted features per window → subject-level LOSO. Establishes the
"how good can simple models get" floor that the CNN/ResNet must beat.

All features are duration-invariant (`src.features`), so the LOSO scores
aren't confounded by the PD-trials-are-longer effect.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)

from tqdm.notebook import tqdm
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, roc_curve

from src.features import per_window_features
from src.data_loader import DATA_ROOT, load_trial


## 1. Load the manifest and recover the post-resize Doppler axis

In [ ]:
manifest = pd.read_csv(ROOT / "outputs" / "preprocessed" / "manifest.csv")
print(f"{len(manifest)} windows across {manifest['subject_id'].nunique()} subjects")

# Pull the doppler axis from one source trial. The preprocessing resize
# step uniformly resamples the doppler dimension, so we rebuild a matching
# linspace at the resized resolution.
sample = manifest.iloc[0]
sample_path = DATA_ROOT / sample["subject_id"] / sample["test"] / sample["trial"] / "stft_data.mat"
sample_doppler = load_trial(sample_path)["doppler"]
arr_shape = np.load(ROOT / "outputs" / "preprocessed" / sample["npy_path"]).shape  # (2, H, W)
H = arr_shape[1]
d_axis = np.linspace(sample_doppler.min(), sample_doppler.max(), H)
print(f"Resized Doppler axis: {H} bins from {d_axis.min():.0f} to {d_axis.max():.0f} Hz")


In [ ]:
feat_root = ROOT / "outputs" / "preprocessed"
feat_rows = []
for r in tqdm(manifest.to_dict("records"), desc="featurising"):
    arr = np.load(feat_root / r["npy_path"])
    feats = per_window_features(arr[0], arr[1], d_axis)
    feats.update({"subject_id": r["subject_id"], "label": r["label"]})
    feat_rows.append(feats)
feat_df = pd.DataFrame(feat_rows)
X = feat_df.drop(columns=["subject_id", "label"]).values
y = feat_df["label"].values
groups = feat_df["subject_id"].values
print(f"X={X.shape}  positives={y.sum()}  subjects={len(set(groups))}")


## 2. LOSO loop — SVM and Random Forest

In [ ]:
def loso(model_factory, X, y, groups):
    subjects = np.array(sorted(set(groups)))
    fold_rows = []
    for s in tqdm(subjects, desc="LOSO"):
        tr = groups != s; va = groups == s
        if y[tr].sum() == 0 or y[tr].sum() == tr.sum():
            continue  # degenerate fold
        pipe = model_factory()
        pipe.fit(X[tr], y[tr])
        if hasattr(pipe, "predict_proba"):
            p = pipe.predict_proba(X[va])[:, 1]
        else:
            p = pipe.decision_function(X[va])
        subj_prob = float(np.mean(p))
        subj_label = int(y[va][0])
        fold_rows.append({"subject_id": s, "label": subj_label,
                          "prob": subj_prob, "pred": int(subj_prob >= 0.5),
                          "n_windows": int(va.sum())})
    return pd.DataFrame(fold_rows)

svm_factory = lambda: Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=1.0, probability=True, random_state=0)),
])
rf_factory = lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)

print("=== SVM ===")
svm_folds = loso(svm_factory, X, y, groups)
print("\n=== Random Forest ===")
rf_folds = loso(rf_factory, X, y, groups)


## 3. Aggregate metrics

In [ ]:
def summarise(folds, name):
    if folds["label"].nunique() < 2:
        print(f"{name}: degenerate (only one class)")
        return
    auc = roc_auc_score(folds["label"], folds["prob"])
    f1 = f1_score(folds["label"], folds["pred"])
    cm = confusion_matrix(folds["label"], folds["pred"])
    print(f"{name}:  subject-AUC={auc:.3f}  F1={f1:.3f}  n={len(folds)}")
    print(f"  Confusion (rows=true 0,1; cols=pred 0,1):\n{cm}")
    return auc

auc_svm = summarise(svm_folds, "SVM")
auc_rf = summarise(rf_folds, "Random Forest")


## 4. ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for folds, name, colour in [(svm_folds, "SVM", "#4c8"),
                             (rf_folds, "Random Forest", "#e66")]:
    if folds["label"].nunique() < 2: continue
    fpr, tpr, _ = roc_curve(folds["label"], folds["prob"])
    auc = roc_auc_score(folds["label"], folds["prob"])
    ax.plot(fpr, tpr, label=f"{name}  AUC={auc:.3f}", color=colour, lw=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Subject-level ROC — LOSO-CV")
ax.legend()
plt.tight_layout(); plt.show()


## 5. Save baseline folds

In [ ]:
out = ROOT / "outputs" / "metrics"
out.mkdir(parents=True, exist_ok=True)
svm_folds.to_csv(out / "baseline_svm_folds.csv", index=False)
rf_folds.to_csv(out / "baseline_rf_folds.csv", index=False)
print("saved baseline_svm_folds.csv and baseline_rf_folds.csv")


### Notes

- Subject-AUC ≠ window-AUC. Window-level signal is usually much higher because windows from the same subject are correlated.
- These features now exclude the duration leak; if the baseline still scores high, it's a real signal — not duration in disguise.
- This is the floor the CNN must clear.
